In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [3]:
!ls


 arav_palsule_hackathon_resume.pdf  'Untitled document (1).gdoc'
 Arav_Resume.pdf		    'Untitled document.gdoc'
'Colab Notebooks'		    'Untitled form.gform'
 image				    'Untitled spreadsheet (1).gsheet'
 samved				    'Untitled spreadsheet.gsheet'
 samved.drawio.png		     wedding_invitation_card.pdf
 sherlock


In [4]:
%cd sherlock

/content/drive/MyDrive/sherlock


In [5]:
!ls


3gab.txt  bosc.txt  croo.txt  gold.txt	maza.txt  redh.txt  silv.txt  vall.txt
3gar.txt  bruc.txt  danc.txt  gree.txt	mems.txt  reig.txt  sixn.txt  veil.txt
3stu.txt  cano.txt  devi.txt  houn.txt	miss.txt  resi.txt  soli.txt  wist.txt
abbe.txt  card.txt  dyin.txt  iden.txt	musg.txt  reti.txt  spec.txt  yell.txt
advs.txt  case.txt  empt.txt  illu.txt	nava.txt  retn.txt  stoc.txt
bery.txt  chas.txt  engr.txt  lady.txt	nobl.txt  scan.txt  stud.txt
blac.txt  cnus.txt  fina.txt  last.txt	norw.txt  seco.txt  suss.txt
blan.txt  copp.txt  five.txt  lion.txt	prio.txt  shos.txt  thor.txt
blue.txt  cree.txt  glor.txt  lstb.txt	redc.txt  sign.txt  twis.txt


In [17]:
import numpy as np
import pandas as pd
import os
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import random
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [18]:
import os

story_path = "/content/drive/MyDrive/sherlock/"

def read_all_stories(story_path):
    txt = []
    for root, _, files in os.walk(story_path):
        for file in files:
            file_path = os.path.join(root, file)
            with open(file_path, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line == '----------':
                        break
                    if line != '':
                        txt.append(line)
    return txt

stories = read_all_stories(story_path)
print("number of lines =", len(stories))

number of lines = 215021


In [19]:
def clean_txt(txt):
    cleaned_txt = []
    for line in txt:
        line = line.lower()
        # Corrected regex to escape double quotes and handle special characters within the character class
        line = re.sub(r"[.,\"!'@#$%^&*(){}?/;`~:<>+=\[\]\\-]", "", line)
        tokens = word_tokenize(line)
        words = [word for word in tokens if word.isalpha()]
        cleaned_txt+=words
    return cleaned_txt

cleaned_stories = clean_txt(stories)
print("number of words = ", len(cleaned_stories))

number of words =  2341418


In [21]:
def make_markov_model(cleaned_stories, n_gram=2):
    markov_model = {}
    for i in range(len(cleaned_stories)-n_gram-1):
        curr_state, next_state = "", ""
        for j in range(n_gram):
            curr_state += cleaned_stories[i+j] + " "
            next_state += cleaned_stories[i+j+n_gram] + " "
        curr_state = curr_state[:-1]
        next_state = next_state[:-1]
        if curr_state not in markov_model:
            markov_model[curr_state] = {}
            markov_model[curr_state][next_state] = 1
        else:
            if next_state in markov_model[curr_state]:
                markov_model[curr_state][next_state] += 1
            else:
                markov_model[curr_state][next_state] = 1

    # calculating transition probabilities
    for curr_state, transition in markov_model.items():
        total = sum(transition.values())
        for state, count in transition.items():
            markov_model[curr_state][state] = count/total

    return markov_model

In [22]:
markov_model = make_markov_model(cleaned_stories)

In [23]:
print("number of states = ", len(markov_model.keys()))

number of states =  215473


In [24]:
print("All possible transitions from 'the game' state: \n")
print(markov_model['the game'])

All possible transitions from 'the game' state: 

{'was up': 0.09009009009009009, 'is afoot': 0.036036036036036036, 'your letter': 0.02702702702702703, 'was whist': 0.036036036036036036, 'in that': 0.036036036036036036, 'the lack': 0.036036036036036036, 'for all': 0.06306306306306306, 'was in': 0.02702702702702703, 'is hardly': 0.02702702702702703, 'would have': 0.036036036036036036, 'is up': 0.06306306306306306, 'is and': 0.036036036036036036, 'in their': 0.036036036036036036, 'may wander': 0.02702702702702703, 'now a': 0.02702702702702703, 'my own': 0.02702702702702703, 'at any': 0.02702702702702703, 'mr holmeswhats': 0.02702702702702703, 'ay whats': 0.02702702702702703, 'my friend': 0.02702702702702703, 'fairly by': 0.02702702702702703, 'is not': 0.02702702702702703, 'was not': 0.02702702702702703, 'was afoot': 0.036036036036036036, 'for the': 0.036036036036036036, 'worth it': 0.02702702702702703, 'you are': 0.02702702702702703, 'i am': 0.02702702702702703, 'now count': 0.0270270270

In [25]:
def generate_story(markov_model, limit=100, start='my god'):
    n = 0
    curr_state = start
    next_state = None
    story = ""
    story+=curr_state+" "
    while n<limit:
        next_state = random.choices(list(markov_model[curr_state].keys()),
                                    list(markov_model[curr_state].values()))

        curr_state = next_state[0]
        story+=curr_state+" "
        n+=1
    return story


In [26]:
for i in range(20):
    print(str(i)+". ", generate_story(markov_model, start="dear holmes", limit=8))

0.  dear holmes you are fortunately the only man whose presence seems essential to the home of my own 
1.  dear holmes said i that this is the smartest of our detective officers being in charge of silas 
2.  dear holmes said i as we drove to the hereford arms where a room one dayit was in 
3.  dear holmes i ejaculated as a bridge for something passing through the trees far away i was naturally 
4.  dear holmes and tell me what you desire to see that and yet this case of blackmailing that 
5.  dear holmes if i wanted your help i have not heard him talk you would not be happy 
6.  dear holmes i exclaimed in unfeigned admiration it is so the young imp can not be unconnected with 
7.  dear holmes what do you make the masks we shall have some hopes ah thank heaven that you 
8.  dear holmes said i thought that young smith was already out of hell that i should be very 
9.  dear holmes i have asked mr sherlock holmes the adventure of the community from that time i do 
10.  dear holmes if i wer

In [27]:
for i in range(20):
    print(str(i)+". ", generate_story(markov_model, start="my dear", limit=8))

0.  my dear holmes what is the best while this conversation and a half long if you are able 
1.  my dear watson do you remember monday was an exceedingly interesting case there is nothing so that it 
2.  my dear young lady my dear watson we are all in your house you think that the world 
3.  my dear fellow and yet i was at my wits end hunter will you let me have the 
4.  my dear watson you have one or two that i am able to give him an opening which 
5.  my dear watson we are too intimately concerned with politics for he will find me a good turn 
6.  my dear watsonhe propped his testtube in the trainers ear he started that he had learned about mrs 
7.  my dear watson i went there and i would refer you to open all the time though i 
8.  my dear holmes i feel that you are miss morstan or of an undertakers mute yet in spite 
9.  my dear fellow it is nearly five now in two bundles of letters and poured them all into 
10.  my dear madam said holmes tell us then how cadogan west met his end i

In [28]:
for i in range(20):
    print(str(i)+". ", generate_story(markov_model, start="i would", limit=8))

0.  i would there was this feeling of an english provincial town his face was clearly reflected in the 
1.  i would not leave it night or day a second consideration struck me jonathan small must have felt 
2.  i would repeat in this way the instant that my judgment may survive the ordeal but you look 
3.  i would willingly do so but there has i fear that it could only be his wife in 
4.  i would give it to you to be in authority been attracted to the description of this man 
5.  i would have knocked him down as he spoke and a look of something over four yards there 
6.  i would stay to show you the road was clearit is never to let mr hosmer angel mr 
7.  i would bring him a certain firmness of jaw and grim tightness about the former to my wife 
8.  i would have shown extraordinary energy in action that i had left so short mahomet singh and give 
9.  i would have trusted your wife knew no sir i know where he lived at sydenham a fortnight 
10.  i would never rest until it led me to belie

In [29]:
print(generate_story(markov_model, start="the case", limit=100))

the case as they came up to her aided and abetted by this german rode off upon a slip of brass on which the complex life of london so well apart form this cigaretteend was it then i will take some hours ago when the sun full upon her face a mantle drawn round her chin her breath came quick and fast and every minute now is of no consequence returned my companion bitterly the question in my own mind it was a relief to him it was i rang the bell watson and if you would only have been done in an instant regret it it was on these occasions i have noticed that fact what does she do and into her sittingroom which was buttoned up for it and just to the right track the idea of this schoolmaster to be accounted for so long a period and so does my wife had gone so far as to poor lestrades discovery it was simply what the porter described as a trivial example of observation and deduction which i usually leave to mary jane she is incorrigible and my duty ill do i was so frightened by their violence